# Tech YouTube Sentiment RAG Model
## YouTube Data Collection Pipeline

**Purpose:** Fetches tech-review videos and their comments from YouTube using the YouTube Data API v3, then stores them in MongoDB Atlas for downstream NLP processing.

**Pipeline Overview:**
| Section | Stage | Output |
|---------|-------|--------|
| 1 | Setup & Dependencies | spaCy model installed |
| 2 | Configuration | API keys & settings loaded |
| 3 | MongoDB Connection | Collection handles |
| 4 | YouTube Video Collection | `youtube_videos` in MongoDB |
| 5 | YouTube Comment Collection | `youtube_comments` in MongoDB |

## Setup
1. Copy `config.template.py` → `config.py` and fill in your credentials
2. Run **Section 1** once to install the spaCy model
3. Run sections **2 → 5** in order

## Requirements
- Python 3.10+
- MongoDB Atlas (free tier works)
- YouTube Data API v3 key(s) — multiple keys recommended to avoid quota limits


## Section 1 — Setup & Dependencies
Installs the spaCy English language model required for downstream NLP processing. Run this cell **once** before running any other section.

In [ ]:
# Download spaCy English model (run once)
import subprocess
import sys
subprocess.run([sys.executable, "-m", "spacy", "download", "en_core_web_sm"], check=True)
print("spaCy model ready.")


## Section 2 — Configuration
Loads all API keys, MongoDB credentials, collection names, and model settings from `config.py`. Keep `config.py` in the **same directory** as this notebook and never commit it to version control.

In [ ]:
import sys, os

# Add the project folder to path so config.py is importable
sys.path.insert(0, os.path.dirname(os.path.abspath("__file__")))

from config import (
    MONGO_URI, MONGO_DB_NAME,
    COLLECTION_VIDEOS, COLLECTION_COMMENTS, COLLECTION_TRANSLATED,
    COLLECTION_TOPIC, COLLECTION_SENTIMENT, COLLECTION_FINAL,
    YOUTUBE_API_KEYS,
    SENTIMENT_MODEL, SUMMARIZATION_MODEL, SPACY_MODEL
)

print("Configuration loaded successfully.")


## Section 3 — MongoDB Connection
Connects to MongoDB Atlas using the URI from `config.py` and initialises handles for all collections used across the pipeline:
- `youtube_videos` — stores video metadata
- `youtube_comments` — stores raw comments
- `youtube_translated_comments_main` — stores translated comments (used by the NLP pipeline)
- `youtube_topic_model`, `youtube_Sentiment_Analysis`, `youtube_final_data` — used by later pipeline stages

In [ ]:
from pymongo import MongoClient

def get_db():
    """Connect to MongoDB Atlas and return (client, db) tuple."""
    client = MongoClient(MONGO_URI)
    db = client[MONGO_DB_NAME]
    return client, db

client, db = get_db()

# Initialise collection handles
videos_collection     = db[COLLECTION_VIDEOS]
comments_collection   = db[COLLECTION_COMMENTS]
translated_collection = db[COLLECTION_TRANSLATED]
topic_collection      = db[COLLECTION_TOPIC]
sentiment_collection  = db[COLLECTION_SENTIMENT]
final_collection      = db[COLLECTION_FINAL]

print("MongoDB Connected!")
print("Available databases:", client.list_database_names())
print("Collections in db  :", db.list_collection_names())


## Section 4 — YouTube Video Collection
Fetches tech-review videos from YouTube using the YouTube Data API v3 and stores their metadata in MongoDB.

**Key design decisions:**
- Multiple API keys are rotated automatically to avoid per-key quota exhaustion
- The discovery document is fetched once and reused to avoid repeated network calls
- Exhausted keys are tracked within the session and skipped on subsequent requests
- Video metadata stored: `video_id`, `title`, `channel`, `view_count`, `like_count`, `comment_count`, `published_at`

### 4.1 — Define Video Fetching Functions
Builds the YouTube API client with key rotation, defines the search and metadata fetch functions, and implements retry logic for quota errors.

In [ ]:
import sys, os
import random
import time
import requests
from googleapiclient.discovery import build_from_document
from googleapiclient.errors import HttpError
 
# --- IMPORT CONFIG ---
sys.path.insert(0, os.path.dirname(os.path.abspath("__file__")))
from config import YOUTUBE_API_KEYS
 
# --- Fetch discovery doc once and reuse ---
_DISCOVERY_URL = "https://www.googleapis.com/discovery/v1/apis/youtube/v3/rest"
_discovery_doc = requests.get(_DISCOVERY_URL).text
 
# --- Track exhausted keys this session ---
exhausted_keys = set()
 
 
def get_youtube_client():
    """Return a (key, client) tuple using a random non-exhausted API key."""
    available = [k for k in YOUTUBE_API_KEYS if k not in exhausted_keys]
    if not available:
        raise RuntimeError("❌ All YouTube API keys have exceeded their quota.")
    key = random.choice(available)
    client = build_from_document(_discovery_doc, developerKey=key)
    return key, client
 
 
def fetch_tech_review_videos(query: str, max_results: int = 20) -> list:
    """
    Fetch YouTube tech-review videos for a search query.
    - Automatically rotates API keys on quota exceeded (403)
    - Skips gracefully on 404 or other errors
    - Stores new videos in MongoDB
    - Returns list of video dicts
    """
    while True:
        try:
            current_key, youtube = get_youtube_client()
 
            # Step 1 — search for video IDs
            search_response = youtube.search().list(
                q=query,
                part="snippet",
                type="video",
                maxResults=max_results,
                order="relevance"
            ).execute()
 
            video_ids = [
                item["id"]["videoId"]
                for item in search_response.get("items", [])
            ]
            if not video_ids:
                print(f"  ⚠️  No results for: {query}")
                return []
 
            # Step 2 — fetch detailed stats
            details_response = youtube.videos().list(
                part="snippet,statistics",
                id=",".join(video_ids)
            ).execute()
 
            # Step 3 — store in MongoDB
            video_data = []
            for item in details_response.get("items", []):
                entry = {
                    "video_id"     : item["id"],
                    "title"        : item["snippet"]["title"],
                    "channel"      : item["snippet"]["channelTitle"],
                    "published_at" : item["snippet"]["publishedAt"],
                    "description"  : item["snippet"]["description"],
                    "view_count"   : int(item["statistics"].get("viewCount",   0)),
                    "like_count"   : int(item["statistics"].get("likeCount",   0)),
                    "comment_count": int(item["statistics"].get("commentCount", 0)),
                }
                if not videos_collection.find_one({"video_id": entry["video_id"]}):
                    videos_collection.insert_one(entry)
                video_data.append(entry)
 
            print(f"  ✅ {len(video_data)} videos stored for: {query}")
            return video_data
 
        except HttpError as e:
            if e.resp.status == 403 and "quotaExceeded" in str(e):
                print(f"  ⚠️  Quota exceeded on key ...{current_key[-8:]}. Rotating...")
                exhausted_keys.add(current_key)
                remaining = len(YOUTUBE_API_KEYS) - len(exhausted_keys)
                print(f"  🔑 Keys remaining: {remaining}/{len(YOUTUBE_API_KEYS)}")
                if not remaining:
                    raise RuntimeError("❌ All YouTube API keys exhausted. Stopping.")
                time.sleep(2)   # brief pause before retrying with next key
            elif e.resp.status == 404:
                print(f"  ⚠️  Not found, skipping: {query}")
                return []
            else:
                print(f"  ❌ HTTP {e.resp.status} error for '{query}': {e}")
                return []
 
        except RuntimeError:
            raise   # bubble up — all keys exhausted
 
        except Exception as e:
            print(f"  ❌ Unexpected error for '{query}': {e}")
            return []

### 4.2 — Run Video Collection
Defines the search queries covering all major tech products (Apple, Samsung, Google, OnePlus, Xiaomi, etc.) and runs the collection loop, storing new videos to MongoDB while skipping duplicates.

In [ ]:
# ── Search queries covering major tech products ──────────────
QUERIES = [

    # Google — Pixel Phones (Flagship)
    "Google Pixel 8 review",

 
"""
    # Apple — iPhones
    "iPhone 17 Pro Max review", "iPhone 17 Pro review",
    "iPhone 17 review", "iPhone Air review", "iPhone 17e review",
    "iPhone 16 Pro Max review", "iPhone 16 Pro review",
    "iPhone 16 review", "iPhone 16 Plus review", "iPhone 16e review",
    "iPhone 15 Pro Max review", "iPhone 15 Plus review", "iPhone 14 Pro review",
    "iPhone SE 4 review","iPhone SE 3 review","iphone 14 review","iPhone 14 Pro Max review",
 
    # Apple — MacBooks & Mac
    "MacBook Air M4 review", "MacBook Pro M5 review",
    "MacBook Pro M4 review", "MacBook Neo review",
    "Mac mini M4 review", "Mac Studio M4 Max review",
    "iMac M4 review",
 
    # Apple — iPad
    "iPad Pro M5 review", "iPad Air M4 review",
    "iPad Air M3 review", "iPad mini A17 Pro review",
    "iPad 11th generation review",

    # Samsung — Galaxy S Series
    "Samsung Galaxy S26 Ultra review", "Samsung Galaxy S26+ review", "Samsung Galaxy S26 review",
    "Samsung Galaxy S25 Ultra review", "Samsung Galaxy S25+ review", "Samsung Galaxy S25 review",
    "Samsung Galaxy S24 review",

    # Google — Pixel Phones (Flagship)
    "Google Pixel 10 Pro XL review", "Google Pixel 10 Pro review",
    "Google Pixel 10 review", "Google Pixel 10 Pro Fold review",
    "Google Pixel 9 Pro XL review", "Google Pixel 9 Pro review",
    "Google Pixel 9 review",

    # Apple — Audio & Accessories
    "AirPods Pro 3 review", "AirPods 4 review",
    "AirPods Max 2 review", "Apple Vision Pro M5 review",
    "Apple Pencil Pro review", "AirTag 2 review",

    # Samsung — Foldables
    "Samsung Galaxy Z Fold 7 review", "Samsung Galaxy Z Flip 7 review",
    "Samsung Galaxy Z Fold 6 review", "Samsung Galaxy Z Flip 6 review",
    "Samsung Galaxy Z Fold 6 Special Edition review",
    "Samsung Galaxy Z TriFold review",
 
    # Samsung — A Series (Mid-range)
    "Samsung Galaxy A56 review", "Samsung Galaxy A36 review", "Samsung Galaxy A26 review",
    "Samsung Galaxy A55 review", "Samsung Galaxy A35 review",
 
    # Samsung — Tablets
    "Samsung Galaxy Tab S10 Ultra review", "Samsung Galaxy Tab S10 Plus review",
    "Samsung Galaxy Tab S10 FE review",

    # Apple — Watch
    "Apple Watch Series 11 review", "Apple Watch Ultra 3 review",
    "Apple Watch SE 3 review", "Apple Watch Series 10 review",
 
    # Samsung — Watch & Ring
    "Samsung Galaxy Watch 8 review", "Samsung Galaxy Watch 8 Ultra review",
    "Samsung Galaxy Watch 7 review", "Samsung Galaxy Watch Ultra review",
    "Samsung Galaxy Ring 2 review", "Samsung Galaxy Ring review",
 
    # Samsung — Buds & Laptops
    "Samsung Galaxy Buds 4 Pro review", "Samsung Galaxy Buds 3 Pro review",
    "Samsung Galaxy Buds 3 review",
    "Samsung Galaxy Book6 Ultra review", "Samsung Galaxy Book6 Pro review",
    "Samsung Galaxy Book6 review",
 
    # Google — Pixel A Series (Mid-range)
    "Google Pixel 10a review", "Google Pixel 9a review", "Google Pixel 8a review",
 
    # Google — Pixel Watch
    "Google Pixel Watch 4 review", "Google Pixel Watch 3 review",
 
    # Google — Pixel Buds
    "Google Pixel Buds Pro 2 review", "Google Pixel Buds 2a review",
    "Google Pixel Buds Pro review",
 
    # Google — Pixel Tablet
    "Google Pixel Tablet 2 review", "Google Pixel Tablet review",
 
    # OnePlus
    "OnePlus 15 review", "OnePlus 15R review",
    "OnePlus 13 review", "OnePlus 13R review",
    "OnePlus Open 2 review", "OnePlus Nord 4 review", "OnePlus Nord CE 4 review",
    "OnePlus Ace 5 Pro review", "OnePlus Watch 3 review",
 
    # Xiaomi
    "Xiaomi 17 Ultra review", "Xiaomi 17 Pro review", "Xiaomi 17 review",
    "Xiaomi 15 Ultra review", "Xiaomi 15 Pro review", "Xiaomi 15 review",
    "Xiaomi Mix Flip 2 review", "Xiaomi Mix Fold 4 review",
    "Xiaomi Redmi Note 14 Pro Plus review", "Xiaomi Redmi Note 13 Pro Plus review",
    "Xiaomi Pad 7 Pro review", "Xiaomi Pad 7 review",
 
    # Oppo
    "Oppo Find X9 Pro review", "Oppo Find X9 Ultra review",
    "Oppo Find X8 Pro review", "Oppo Find X8 Ultra review",
    "Oppo Reno 14 Pro review", "Oppo Find N5 foldable review",
 
    # Realme
    "Realme GT 8 Pro review", "Realme GT 7 Pro review",
    "Realme GT 6 review", "Realme 14 Pro Plus review",
    "Realme 15 Pro review",
"""

]
 
# --- RUN ---
print(f"Starting fetch for {len(QUERIES)} queries...\n")
 
for i, query in enumerate(QUERIES, start=1):
    print(f"[{i}/{len(QUERIES)}] {query}")
    try:
        fetch_tech_review_videos(query, max_results=1)
    except RuntimeError as e:
        print(f"\n{e}")
        print("Stopping early — all keys exhausted.")
        break
    time.sleep(1)   # be polite to the quota
 
print(f"\n✅ Total unique videos in DB: {len(videos_collection.distinct('video_id'))}")

## Section 5 — YouTube Comment Collection
For each video stored in MongoDB, fetches top-level comments using the YouTube Data API v3 and saves them to the `youtube_comments` collection.

**Key design decisions:**
- Comments are filtered using `UX_KEYWORDS` — only comments that indicate real user experience (ownership, usage, upgrade decisions) are kept, reducing noise
- A checkpoint file (`checkpoint.txt`) tracks the last processed video index so the run can be resumed if interrupted
- A 2-second delay between requests prevents quota exhaustion
- `BATCH_LIMIT` caps how many videos are processed per run, allowing controlled incremental collection

### 5.1 — Define Comment Fetching Functions & UX Keyword Filter
Defines the `UX_KEYWORDS` list used to filter comments, the API key rotation logic, and the comment fetch function with pagination support.

In [ ]:
import sys, os
import random
import time
import requests
from googleapiclient.discovery import build_from_document
from googleapiclient.errors import HttpError
# --- Keywords that indicate a user experience comment ---
UX_KEYWORDS = [
    # ownership / usage — with AND without "i"
    "i bought", "bought",
    "i purchased", "purchased",
    "i own", "i owned", "owned",
    "i have", "i've had",
    "i got", "got it",
    "i use", "i used", "using it",
    "i switched", "switched",
    "i upgraded", "upgraded",
    "i returned", "returned it",
    "i tried", "tried it",
    "i tested", "tested it",
    "i received", "received",
    "i ordered", "ordered",
    "i picked up", "picked up",
    "i replaced", "replaced",
    "i sold", "sold it",
    "i traded", "traded",
    "i gifted", "gifted",
    "i borrowed", "borrowed",
    "we bought", "we use",
    "my wife", "my husband", "my family",
    "my phone", "my device", "my unit", "my tablet", "my watch",
 
    # time-based experience
    "after", "months", "weeks", "days", "years",
    "1 month", "2 months", "3 months", "6 months", "1 year",
    "day one", "first day", "first week", "long term", "daily use",
    "everyday", "daily driver",
 
    # hardware — general
    "battery", "battery life", "charging", "charger", "fast charging",
    "wireless charging", "battery drain", "battery dead", "dies fast",
    "camera", "photo", "photos", "video", "videos", "selfie",
    "zoom", "night mode", "portrait mode", "ultra wide",
    "screen", "display", "amoled", "oled", "lcd", "refresh rate",
    "brightness", "sunlight", "resolution", "notch", "punch hole",
    "processor", "chip", "chipset", "performance", "speed",
    "ram", "storage", "memory", "space",
    "speaker", "speakers", "audio", "sound", "volume", "microphone",
    "fingerprint", "face id", "face unlock", "biometric",
    "port", "usb", "headphone jack", "sim", "esim",
    "build quality", "build", "design", "feel", "weight", "size",
    "glass", "metal", "plastic", "frame", "back", "grip",
    "waterproof", "water resistant", "ip68", "ip67", "drop",
    "scratch", "cracked", "broken", "durable", "sturdy", "fragile",
 
    # software / OS
    "update", "software", "os", "android", "ios", "one ui", "miui",
    "bug", "bugs", "issue", "issues", "problem", "problems", "fix",
    "crash", "crashes", "freezes", "restart", "reboot", "brick",
    "bloatware", "ads", "ui", "ux", "interface", "settings",
    "feature", "features", "missing", "added", "removed",
 
    # performance issues
    "lag", "lagging", "slow", "sluggish", "fast", "smooth", "snappy",
    "overheating", "heating", "hot", "warm", "throttle", "throttling",
    "stutter", "stuttering", "frame drop", "glitch", "glitches",
 
    # sentiment / opinion
    "worth", "not worth", "value", "value for money", "price",
    "expensive", "cheap", "affordable", "overpriced", "budget",
    "disappointed", "disappointing", "frustrating", "frustrated",
    "impressed", "impressive", "satisfied", "happy", "unhappy",
    "amazing", "awesome", "excellent", "fantastic", "outstanding",
    "terrible", "horrible", "awful", "bad", "poor", "mediocre",
    "love", "loved", "hate", "hated", "like", "dislike",
    "best", "worst", "better", "worse", "good", "great",
    "recommend", "don't recommend", "would recommend", "not recommended",
    "regret", "regrets", "mistake", "perfect", "flawless",
    "overrated", "underrated", "hyped", "overhyped",
 
    # comparison
    "compared to", "vs", "versus", "better than", "worse than",
    "upgrade from", "downgrade", "switched from", "coming from",
    "previous", "last year", "older model", "new model",
    "iphone", "samsung", "pixel", "oneplus", "xiaomi", "huawei",
 
    # repair / support
    "repair", "replaced", "warranty", "service center", "apple care",
    "screen replaced", "battery replaced", "refund", "return policy",
]


def is_ux_comment(text: str) -> bool:
    """Return True if comment contains at least one user experience keyword.
    Case-insensitive — handles ALL CAPS, lowercase, and mixed case."""
    text_lower = text.lower()
    return any(kw in text_lower for kw in UX_KEYWORDS)


def fetch_comments_for_video(video_id: str, product_name: str) -> list:
    """
    Fetch ALL comments for a single video using pagination.
    Only stores comments that match UX keywords.
    Returns a list of newly inserted comment dicts.
    """
    new_comments = []
    next_page_token = None
    page = 0

    while True:
        try:
            current_key, youtube = get_youtube_client()

            request_kwargs = {
                "part"      : "snippet",
                "videoId"   : video_id,
                "maxResults": 100,          # max allowed per page
                "textFormat": "plainText",
            }
            if next_page_token:
                request_kwargs["pageToken"] = next_page_token

            response = youtube.commentThreads().list(**request_kwargs).execute()
            page += 1

            for item in response.get("items", []):
                snippet = item["snippet"]["topLevelComment"]["snippet"]
                text = snippet["textDisplay"]

                if not is_ux_comment(text):
                    continue  # skip non-UX comments

                entry = {
                    "video_id": video_id,
                    "product" : product_name,
                    "comment" : text,
                    "likes"   : snippet["likeCount"],
                }
                if not comments_collection.find_one({"comment": entry["comment"]}):
                    comments_collection.insert_one(entry)
                    new_comments.append(entry)

            # check if there are more pages
            next_page_token = response.get("nextPageToken")
            if not next_page_token:
                break   # no more pages — done

        except HttpError as e:
            if e.resp.status == 403 and "quotaExceeded" in str(e):
                print(f"  ⚠️  Quota exceeded on key ...{current_key[-8:]}. Rotating...")
                exhausted_keys.add(current_key)
                if not [k for k in YOUTUBE_API_KEYS if k not in exhausted_keys]:
                    raise RuntimeError("❌ All YouTube API keys exhausted.")
                time.sleep(2)   # retry same page with new key
            else:
                print(f"  Error fetching comments for {video_id}: {e}")
                break

        except Exception as e:
            print(f"  Error fetching comments for {video_id}: {e}")
            break

    print(f"  +{len(new_comments)} UX comments ({page} pages) — {product_name[:50]}")
    return new_comments


### 5.2 — Run Comment Collection Loop
Iterates over all stored videos, fetches their comments, applies the UX keyword filter, and saves matching comments to MongoDB. Progress is checkpointed so the loop can resume after interruption.

In [ ]:
# --- Iterate over all stored videos and collect comments ---
all_videos = list(videos_collection.find({}, {"video_id": 1, "title": 1}))
print(f"Processing {len(all_videos)} videos...")

# --- Load last processed index ---
CHECKPOINT_FILE = "checkpoint.txt"
start_index = 0
if os.path.exists(CHECKPOINT_FILE):
    with open(CHECKPOINT_FILE, "r") as f:
        start_index = int(f.read().strip())
    print(f"Resuming from video {start_index + 1}...")

BATCH_LIMIT = 250  # stop after 500 videos

for i, video in enumerate(all_videos, start=1):
    if i <= start_index:
        continue  # skip already processed videos
    if i > start_index + BATCH_LIMIT:
        print(f"\n⏸ Batch of {BATCH_LIMIT} videos done. Stopping.")
        break

    product_name = video["title"].split(" review")[0]
    print(f"[{i}/{len(all_videos)}] {product_name[:60]}")
    try:
        fetch_comments_for_video(video["video_id"], product_name)
    except RuntimeError as e:
        print(f"\n{e}")
        print("Stopping early — all keys exhausted.")
        break
    time.sleep(1)

    # --- Save checkpoint after each video ---
    with open(CHECKPOINT_FILE, "w") as f:
        f.write(str(i))

print(f"\n✅ Total comments in DB: {comments_collection.count_documents({})}")